# **Automatizar Reestructuras**
El objetivo del código es tener una idea en la forma en la que se pueden automatizar las reestructuras para poder evitar errores por parte de los asesores y reducir la carga de trabajo que estas generan.

# **Imports Necesarios**

In [ ]:
import pandas as pd
import gspread
import hashlib
from pathlib import Path
import random
from datetime import datetime
from zoneinfo import ZoneInfo
import json
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from google.oauth2.service_account import Credentials
from gspread.exceptions import APIError
import numpy as np
from time import sleep
import os
import requests
from requests.exceptions import ChunkedEncodingError, ConnectionError, Timeout
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from io import StringIO
# Importamos el diccionario con valores predefinidos
from collections import defaultdict
import re
import io

mesesDict = {
    1: 'Enero',2: 'Febrero',3: 'Marzo',4: 'Abril',5: 'Mayo',6: 'Junio',
    7: 'Julio',8: 'Agosto',9: 'Septiembre',10: 'Octubre',11: 'Noviembre',12: 'Diciembre'
}

# Funcion Auxiliar para obtener variables de entorno/ secretos de forma robusta
def getEnvVar(var: str):
  try: # Si el entorno es Colab las credenciales se obtendrán de esa forma
    from google.colab import userdata
    return userdata.get(var)
  except ImportError: # Si el entorno es GitHub las credenciales se obtienen es mediante variables de entorno
    creds = os.getenv(var)
    if not creds:
      raise ValueError("No se encontraron las credenciales en las variables de entorno.")
    return creds.strip()

# Función Auxiliar para reintentar cualquier llamda al API de sheets
def _retry(fn, label="", tries=10, base_sleep=1.5, jitter=0.6, max_sleep=45):
    RETRIABLE_CODES = ["[500]", "[502]", "[503]", "[504]", "[429]"]
    last_err = None
    for i in range(tries):
        try:
            return fn()
        except APIError as e:
            last_err = e
            msg = str(e)
            # Si el error fue del servidor
            if any(c in msg for c in RETRIABLE_CODES):
                sleep_s = min((base_sleep) * (2 ** i) + np.random.uniform(0, jitter), max_sleep)
                print(f"[RETRY {i+1}/{tries}] {label} -> {msg[:120]}... sleep {sleep_s:.1f}s")
                # Esperamos para no saturar al API
                sleep(sleep_s)
                continue
            raise
    raise last_err

# Función Auxiliar para imputar NaNs
def imputeNans(df: pd.DataFrame, col: str, value):
  # Se define la mascara de valores nulos
  mask = df[col].isna()
  # A los valores nulos se aplica el valor
  df.loc[mask, col] = value

def print_grid(strings, n=5, sep= ' / '):
    """
    Prints a list of strings with 'n' items per row,
    separated by 'sep'.
    """
    for i, word in enumerate(strings):
        # Print the word. Use end='' to manage the separator manually.
        print(word, end="")

        # Check if we need a separator or a newline
        if (i + 1) % n == 0:
            print()  # Newline after every n-th element
        elif i < len(strings) - 1:
            print(sep, end="") # Only print separator if not at the end of a row

    # Add a final newline if the last row wasn't "full"
    if len(strings) % n != 0:
        print()

# Función Auxiliar para obtener credenciales
def getCreds():
  try: # Si el entorno es Colab las credenciales se obtendrán de esa forma
    from google.colab import userdata
    return json.loads(userdata.get('MI_JSON'))
  except ImportError: # Si el entorno es GitHub las credenciales se obtienen es mediante variables de entorno
    creds = os.getenv('MI_JSON')
    if not creds:
      raise ValueError("No se encontraron las credenciales en las variables de entorno.")
    return json.loads(creds.strip())

# 1. Retrieve the secret from Colab
service_account_info = getCreds()

# 2. Define the scope and authenticate
scope = ['https://www.googleapis.com/auth/spreadsheets',
         'https://www.googleapis.com/auth/drive']

creds = Credentials.from_service_account_info(service_account_info, scopes=scope)
client = gspread.authorize(creds)

# Creamos el Servicio de Drive
today = pd.Timestamp.now()

# 1. **Obtención de Datos**
Los Datos que se Requieren son:
- **Mensualidades de Mutatio**
- **Flujo de Cartera Berex**
- **Datos de Moras**

**Nota:** Se están dejando los datos de la última Originación por Fecha_Origen

## 1.0 **Cambios de Referencia**

In [ ]:
# Abrimos la SpreadSheet
refChangesShId = '1jcPPhtF2YK3Kr7P_A0Mgh2OqhOfnVWB2to3UPoSH5tE'
refChangesSH = client.open_by_key(refChangesShId)
# Aca se cargan los cambios de referencia
refChangesWS = refChangesSH.worksheet('Cambios de Referencia')
records = refChangesWS.get_all_values()

## La llave sera la referencia vieja y el valor la referencia nueva
if len(records)>0 and len(records[0])>1:
  refChangesDict = {str(row[0]).replace('.0','').strip():str(row[1]).replace('.0','').strip() for row in records}
else:
  refChangesDict = {}

## 1.1 **Datos de Moras (✅Check)**
Datos que se obtienen: **morasDF**
- Referencia
- Fecha
- Fecha_Origen
- Por_Cobrar
- Pago
- Status_Mora

In [ ]:
#### Traemos los Datos desde Sheets
# Abrimos la SpreadSheet
morasShId = '1jcPPhtF2YK3Kr7P_A0Mgh2OqhOfnVWB2to3UPoSH5tE'
morasSH = client.open_by_key(morasShId)
# Abrimos la WorkSheet
morasWsName = 'Comisión'
morasWs = morasSH.worksheet(morasWsName)
# Obtenemos los datos
morasDF = _retry(lambda: get_as_dataframe(morasWs, evaluate_formulas=True))

# Dejamos las columnas Requeridas
moraCols = ['REFERENCIA','FECHA','FECHA_ORIGEN','X_COBRAR','PAGO','MORA_STATUS']
morasDF = morasDF[moraCols]

# Renombrar las Columnas a nombres más amigables
morasDF = morasDF.rename(columns={'REFERENCIA':'Referencia','FECHA':'Fecha',
                                  'FECHA_ORIGEN':'Fecha_Origen','X_COBRAR':'Por_Cobrar',
                                  'PAGO':'Pago','MORA_STATUS':'Status_Mora'})

# Volvemos la Columna Referencia a string
morasDF['Referencia'] = morasDF['Referencia'].apply(lambda x: str(x).replace('.0','').strip())

# Aplicamos el Cambio de Referencia
morasDF['Referencia'] = morasDF['Referencia'].apply(lambda x: refChangesDict.get(x,x))

# Volvemos la Columna Fecha_Origen y Fecha a Datetime
morasDF['Fecha_Origen'] = pd.to_datetime(morasDF['Fecha_Origen'], errors='coerce', dayfirst=False)
morasDF['Fecha'] = pd.to_datetime(morasDF['Fecha'], errors='coerce', dayfirst=False)

### Ahora la idea es asignar la Fecha_Origen correcta a todos los requeridos
# Se realiza ordenando por Referencia y Fecha
# Así se guardan los índices hasta que se encuentre la siguiente fecha de originacion
# Mientras no exista una fecha de originación se van agregando los índices
# Cuando aparezca una fecha de originación nueva los indices se actualizan por la fecha de originación

# 1. Ordenar por Referencia y Fecha
morasDF = morasDF.sort_values(by=['Referencia', 'Fecha'])

# 2. Iterar por cada una de las filas
# Definimos la currFechaOrigen como nula, la referencia actual como nula y los indices como vacios
currFechaOrigen = None
indices = []
for idx, row in morasDF.iterrows():
    # 3. Verificar si es existe una fecha de origen (no es NaN)
    if not pd.isna(row['Fecha_Origen']):
      # 4. Actualizar los índices si hay
      if indices:
        morasDF.loc[indices, 'Fecha_Origen'] = currFechaOrigen
      # 5. Actualizar la fecha de origen que se lleva hasta el momento
      currFechaOrigen = row['Fecha_Origen']
      currRef = row['Referencia']
      # Reiniciamos los Indices
      indices = [idx]
      # Se continua con la ejecucion
      continue
    # 6. Si no existe una fecha de origen se añade el indice a la lista
    indices.append(idx)
if indices:
  # Se realiza la actualizacion final
  morasDF.loc[indices, 'Fecha_Origen'] = currFechaOrigen

# Ahora se vuelven las columnas Por_Cobrar y Pago a numerico
morasDF['Por_Cobrar'] = pd.to_numeric(morasDF['Por_Cobrar'], errors='coerce')
morasDF['Pago'] = pd.to_numeric(morasDF['Pago'], errors='coerce')

# Ahora se imputan los Nans de Por_Cobrar y Pago a 0
imputeNans(morasDF, 'Por_Cobrar', 0)
imputeNans(morasDF, 'Pago', 0)

# Dejamos los Últimos Datos por Referencia y Fecha_Origen
# 1. Obtenemos la Máxima Fecha_Origen por Referencia
maxFechaOrigen = morasDF.groupby('Referencia')['Fecha_Origen'].transform('max')
# 2. Dejamos datos de esa Última Fecha_Origen por Referencia
morasDF = morasDF[morasDF['Fecha_Origen'] == maxFechaOrigen]


# Quitamos Referencias Cerradas (Status_Mora == 'Cerrado')
refsCerradas = morasDF[morasDF['Status_Mora'] == 'Cerrado']['Referencia'].tolist()
morasDF = morasDF[~morasDF['Referencia'].isin(refsCerradas)]

print('✅Moras cargadas con éxito, columnas: {}'.format(', '.join(morasDF.columns.tolist())))
print('🚼Filas totales: {} filas.'.format(len(morasDF)))

✅Moras cargadas con éxito, columnas: Referencia, Fecha, Por_Cobrar, Pago, X_COBRAR_FUTURO, PLAN_PAGADO, PLAN_PAGADO_INCOBRABLE, Fecha_Origen, FECHA_COBRO, RESPONSABLE, FECHA_DE_PAGO, MOROSO, DIAS_EN_MORA, MORA_STATUS, Saldo actual, PAGO_PARCIAL
🚼Filas totales: 17539 filas.


## 1.2 **Datos de Berex (✅Check)**
Datos que se Obtienen: **carteraDF**
- Referencia
- Fecha_Origen
- Fecha_Pago_Berex
- Monto_Berex
- Destino

In [ ]:
#### Traemos los Datos desde Sheets
# Abrimos la SpreadSheet
carteraShId = '13Vf32LzRI2V95dIUqfevzm-ZmsDR3d17UTre_7XJ-UU'
carteraSH = client.open_by_key(carteraShId)
# Abrimos la Worksheet
carteraWsName = 'Cartera'
carteraWs = carteraSH.worksheet(carteraWsName)
# Obtenemos los datos
carteraDF = _retry(lambda: get_as_dataframe(carteraWs, evaluate_formulas=True))

# Adicionalmente, vamos a cargar la Worksheet 'Tradicional'
tradWs = carteraSH.worksheet('Tradicional')
# Obtenemos los datos
tradDF = _retry(lambda: get_as_dataframe(tradWs, evaluate_formulas=True))

# Creamos el Diccionario de Cambio de Nombres de Columnas
changeColsDict = {'reference':'Referencia','Originado':'Fecha_Origen',
                'payment_date':'Fecha_Pago_Berex','amount':'Monto_Berex',
                'destination':'Destino'}

# Renombramos las Columnas a Nombres más entendibles
carteraDF = carteraDF.rename(columns=changeColsDict)
tradDF = tradDF.rename(columns=changeColsDict)

# Dejamos las Columnas Requeridas
carteraCols = ['Referencia','Fecha_Origen','Fecha_Pago_Berex','Monto_Berex','Destino']
carteraDF = carteraDF[carteraCols]
tradDF = tradDF[carteraCols]

# Volvemos la Columna Referencia a string
carteraDF['Referencia'] = carteraDF['Referencia'].apply(lambda x: str(x).replace('.0','').strip())
tradDF['Referencia'] = tradDF['Referencia'].apply(lambda x: str(x).replace('.0','').strip())

# Aplicamos los Cambios de Referencia
carteraDF['Referencia'] = carteraDF['Referencia'].apply(lambda x: refChangesDict.get(x,x))
tradDF['Referencia'] = tradDF['Referencia'].apply(lambda x: refChangesDict.get(x,x))

# Filtramos a tradDF donde la Referencia se encuentre en CarteraDF
carteraRefs = carteraDF['Referencia'].tolist()
tradDF = tradDF[tradDF['Referencia'].isin(carteraRefs)]

# Unimos carteraDF y tradDF
carteraDF = pd.concat([carteraDF, tradDF], ignore_index=True, join='outer')

# Convertimos la Columna Fecha_Origen y Fecha_Pago_Berex a Datetime
carteraDF['Fecha_Origen'] = pd.to_datetime(carteraDF['Fecha_Origen'], errors='coerce', dayfirst=True)
carteraDF['Fecha_Pago_Berex'] = pd.to_datetime(carteraDF['Fecha_Pago_Berex'], errors='coerce', dayfirst=True)

# Cambiamos los Datos donde Destino == 'bank' a 'Banco'
carteraDF.loc[carteraDF['Destino'].str.lower() == 'bank', 'Destino'] = 'Banco'
# Cambimos los Datos donde Destino == 'commission' a 'Comision'
carteraDF.loc[carteraDF['Destino'].str.lower() == 'commission', 'Destino'] = 'Comision'

print('✅Cartera cargada con éxito, columnas: {}'.format(', '.join(carteraDF.columns.tolist())))
print('🚼Filas totales: {} filas.'.format(len(carteraDF)))

✅Cartera cargada con éxito, columnas: Referencia, id, amount, bank, debt_id, destination, Fecha_Pago_Berex, payment_number, requester, Fecha_Origen, Unnamed: 10
🚼Filas totales: 65898 filas.


## 1.3 **Datos de Mutatio (Mensualidades) (Check ✅)**
Datos que se Obtienen: **mensualidadesDF**
- Referencia
- Status_Facturacion
- Status_Reparadora
- Monto_Mensualidad
- Fecha_Cobro

### 1.3.0 **Definir Inputs y Clases**

In [ ]:
# Definimos la URL a la cúal hacer la request
BASE_URL = "https://mutatio-api.gobravo.dev"

# Obtenemos Credenciales
USER_API   = getEnvVar("USER_API")
SECRET_API = getEnvVar("SECRET_API")
if not USER_API or not SECRET_API:
    raise ValueError("Falta USER_API o SECRET_API.")

# Función Auxiliar para crear sesión
def makeSession():
    s = requests.Session()
    retry = Retry(
        total=3, connect=2, read=2, backoff_factor=1.2,
        status_forcelist=[502, 503, 504],
        allowed_methods={"POST"},
        raise_on_status=False
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s

# Clase de Autenticación con Refreshing Token
class Auth:
  def __init__(self, user: str, secret: str, baseUrl: str, session):
    self.user = user
    self.secret = secret
    self.baseUrl = baseUrl
    self.session = session
    self.getNewToken()

  # Función Auxiliar para Definir
  def getNewToken(self):
    url = f"{self.baseUrl}/auth/generate-token"
    r = self.session.post(
        url,
        json={"user": self.user, "secret": self.secret},
        headers={"Content-Type":"application/json", "Accept":"application/json"},
        timeout=(10, 45),
    )
    r.raise_for_status()
    tok = r.json().get("token")
    if not tok:
        raise RuntimeError("Auth OK pero no vino 'token'.")
    self.token = tok

  # Función Auxiliar para Obtener un Nuevo Token de Autenticación
  def refresh(self):
    self.getNewToken()
    return self.token

### 1.3.1 **Obtención de Datos**

In [ ]:
# Creamos la Sesión
session = makeSession()

# Creamos la Autenticación
auth = Auth(USER_API, SECRET_API, BASE_URL, session)

# --- Descarga robusta de UNA página ---
def downloadPage(page: int, filtros: list, extra_params: dict | None = None,
                 max_retries_401: int = 2, max_retries_5xx: int = 6,
                 base_sleep: float = 0.8):
    """
    Devuelve (df, headers, raw_bytes).
    - 401 -> refresca token y reintenta (hasta max_retries_401).
    - 5xx -> backoff exponencial + jitter (hasta max_retries_5xx).
    Levanta excepción solo si se agotaron los reintentos.
    """
    url = f"{BASE_URL}/accounting/facturations/download"
    params = {"pageToDownload": str(page)}
    if extra_params:
        params.update(extra_params)

    tries_401 = 0
    tries_5xx = 0

    while True:
        headers_req = {
            "Authorization": f"Bearer {auth.token}",
            "Accept": "text/csv",
            "Content-Type": "application/json",
        }

        r = session.post(
            url,
            headers=headers_req,
            params=params,
            json=filtros,
            timeout=(15, 120)
        )

        # 401: token vencido -> refresh y reintentar
        if r.status_code == 401:
            if tries_401 >= max_retries_401:
                r.raise_for_status()
            tries_401 += 1
            print(f"[{page}] 401 recibido. Refrescando token ({tries_401}/{max_retries_401})")
            auth.refresh()
            sleep(0.6)
            continue

        # 5xx transitorios
        if r.status_code in (502, 503, 504):
            if tries_5xx >= max_retries_5xx:
                r.raise_for_status()

            tries_5xx += 1
            sleep_s = base_sleep * (2 ** (tries_5xx - 1)) + random.uniform(0, 0.4)

            print(f"[{page}] {r.status_code} transitorio. Reintentando en {sleep_s:.1f}s "
                  f"({tries_5xx}/{max_retries_5xx})")

            sleep(sleep_s)
            continue

        # Otros errores
        r.raise_for_status()

        raw = r.content or b""

        if raw.strip() == b"":
            return pd.DataFrame(), r.headers, raw

        df = pd.read_csv(io.BytesIO(raw), sep=";", dtype=str)

        return df, r.headers, raw


# --- Descarga TODAS las páginas ---
def downloadAllPages(
    filtros: list,
    max_pages: int = 500,
    extra_params: dict | None = None,
    sleep_between_pages: float = 0.5,
    checkpoint_parquet_prefix: str | None = None,
    start_page: int = 0,
):

    frames = []
    seen_hashes = set()
    total_pages_hint = None

    for page in range(start_page, max_pages):

        try:
            df, hdrs, raw = downloadPage(
                page,
                filtros,
                extra_params=extra_params,
                max_retries_401=2,
                max_retries_5xx=6,
                base_sleep=0.8
            )

        except requests.HTTPError as e:

            status = getattr(e.response, "status_code", None)

            print(f"[{page}] Error definitivo {status}: {e}")
            print(f"Puedes reanudar luego con start_page={page}")

            break

        # Headers informativos
        for k in (
            "X-Total-Count",
            "X-Total-Pages",
            "X-Page-Index",
            "X-Page-Size",
            "Content-Range",
        ):
            if k in hdrs:
                print(f"[h] {k}: {hdrs[k]}")

        if "X-Total-Pages" in hdrs:
            try:
                total_pages_hint = int(hdrs["X-Total-Pages"])
            except:
                pass

        if df.empty:
            print(f"[{page}] Página vacía. Fin.")
            break

        # CSV duplicado
        h = hashlib.md5(raw).hexdigest()

        if h in seen_hashes:
            print(f"🤔[{page}] CSV idéntico a una página anterior. Fin.")
            break

        seen_hashes.add(h)

        print(f"🆔[{page}] Filas: {len(df)} | Columnas: {len(df.columns)}")

        frames.append(df)

        # Guardar checkpoint
        if checkpoint_parquet_prefix:

            path = f"{checkpoint_parquet_prefix}_{page:04d}.parquet"

            df.to_parquet(path, index=False)

            print(f"[{page}] Guardado checkpoint: {path}")

        # Última página
        if len(df) < 100000:
            print(f"ℹ️[{page}] Última página (menor a 100k).")
            break

        if total_pages_hint is not None and (page + 1) >= total_pages_hint:
            print(f"ℹ️[{page}] Alcanzado X-Total-Pages={total_pages_hint}.")
            break

        sleep(sleep_between_pages)

    if not frames:
        return pd.DataFrame()

    fullDataFrame = pd.concat(frames, ignore_index=True)

    print("⚙️Total filas acumuladas:", len(fullDataFrame))

    return fullDataFrame


# --------- Uso ----------

filtrosAplicados = [

    {
        "column": "Comision",
        "attribute": "commissionType",
        "columnType": "ENUM",
        "table": "facturation",
        "operation": "IGUAL_A",
        "value": "MENSUALIDAD_COLOMBIA"
    },

    {
        "column": "Status",
        "attribute": "statusType",
        "columnType": "ENUM",
        "table": "facturation",
        "operation": "IGUAL_A",
        "value": "POR_COBRAR"
    }

]

# Tamaño de página si el backend lo soporta
# extra = {"pageSize": 100000}
# extra = {"size": 100000}
# extra = {"rowsPerPage": 100000}

extra = None

mensualidadesDF = downloadAllPages(
    filtros=filtrosAplicados,
    max_pages=200,
    extra_params=extra,
    sleep_between_pages=0.6,
    checkpoint_parquet_prefix=None,
    start_page=0
)

print(
    "✅Mensualidades Cargadas con éxito, columnas:",
    ", ".join(mensualidadesDF.columns.tolist())
)

print(
    "🚼Filas totales:",
    len(mensualidadesDF)
)

🆔[0] Filas: 100000 | Columnas: 38
🆔[1] Filas: 100000 | Columnas: 38
🆔[2] Filas: 100000 | Columnas: 38
🆔[3] Filas: 100000 | Columnas: 38
🆔[4] Filas: 100000 | Columnas: 38
🆔[5] Filas: 100000 | Columnas: 38
[6] 401 recibido. Refrescando token (intento 1/2)...
🆔[6] Filas: 100000 | Columnas: 38
🆔[7] Filas: 100000 | Columnas: 38
🆔[8] Filas: 79449 | Columnas: 38
ℹ️[8] Última página (menor a 100k).
⚙️Total filas acumuladas: 879449
✅Mensualidades Cargadas con éxito, columnas: Id, Empresa Emisora, Credito, Status facturacion, Tipo de comision, Status reparadora, Referencia, Monto, Fecha de facturacion, Fecha de envio de cobro, Fecha de cobro, Banco receptor, Fecha de operacion, Saldo, Factura electronica, Fecha FE, Folio, Comision Restructurada, Fecha de estructurado, Id Quickbase, Fecha de creacion, Fecha de actualizacion, Creado por, Actualizado por, Cargado en Siesa, Subido a Siesa por, Fecha de carga a Siesa, Folio Nota Credito, Prefijo Resolucion, CUFE, Descripcion, Referencia credito, Tien

### 1.3.2 **Limpieza de Mensualidades**

In [ ]:
# --- LIMPIEZA DE COLUMNAS ---
# Renombramos columnas para mayor entendimiento
mensualidadesDF = mensualidadesDF.rename(columns={
    'Status facturacion':'Status_Facturacion',
    'Fecha de cobro':'Fecha_Cobro',
    'Status reparadora':'Status_Reparadora',
    'Monto': 'Monto_Mensualidad',
    })

# Volvemos la Referencia a String
mensualidadesDF['Referencia'] = mensualidadesDF['Referencia'].apply(lambda x: str(x).replace('.0','').strip())

# Volvemos las Columnas Númericas a Números: Monto_Mensualidad
mensualidadesDF['Monto_Mensualidad'] = pd.to_numeric(mensualidadesDF['Monto_Mensualidad'], errors='coerce')

# Volvemos las Columnas Necesarias a Datetime: Fecha_Cobro
mensualidadesDF['Fecha_Cobro'] = pd.to_datetime(mensualidadesDF['Fecha_Cobro'], errors='coerce', dayfirst=True)

# Renombramos Monto a Monto_Mensualidad
mensualidadesDF = mensualidadesDF.rename(columns={'Monto':'Monto_Mensualidad'})

# --- FILTROS APLICADOS ---
# 1. Status_Facturacion es POR_COBRAR

#mensualidadesDF = mensualidadesDF[mensualidadesDF['Status_Facturacion'] == 'POR_COBRAR']
# 2. Status_Reparadora es ACTIVO (Por Ver)
# mensualidadesDF = mensualidadesDF[mensualidadesDF['Status_Reparadora'] == 'ACTIVO']

# Dejamos Solo las Columnas Necesarias para el Análisis
colsNecesarias = ['Referencia''Status_Facturacion','Status_Reparadora','Monto_Mensualidad',
                'Fecha_Cobro']
mensualidadesDF = mensualidadesDF[colsNecesarias]

print('✅Mensualidades Cargadas con éxito, columnas: {}'.format(', '.join(mensualidadesDF.columns.tolist())))
print('🚼Filas totales: {} filas.'.format(len(mensualidadesDF)))

/tmp/ipykernel_156/679303092.py:21: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  mensualidadesDF['Fecha_Envio_Cobro'] = pd.to_datetime(mensualidadesDF['Fecha_Envio_Cobro'], errors='coerce', dayfirst=True)


✅Mensualidades Cargadas con éxito, columnas: Id, Empresa Emisora, Credito, Status_Facturacion, Tipo_Comision, Status_Reparadora, Referencia, Monto, Fecha_Facturacion, Fecha_Envio_Cobro, Fecha_Cobro, Banco receptor, Fecha de operacion, Saldo, Factura electronica, Fecha FE, Folio, Comision Restructurada, Fecha de estructurado, Id Quickbase, Fecha de creacion, Fecha de actualizacion, Creado por, Actualizado por, Cargado en Siesa, Subido a Siesa por, Fecha de carga a Siesa, Folio Nota Credito, Prefijo Resolucion, CUFE, Descripcion, Referencia credito, Tiene Nota Credito Electronica, Fecha de Nota Credito Electronica, Status NC Sistema Contable, Subido NC a Sistema Contable por, Fecha NC subida a sistema contable, Tipo de Documento Electronico
🚼Filas totales: 13004 filas.


# 2. **Filtrado de Referencias**

## 2.1 **Cambios de Referencia**

In [ ]:
# Actualizamos las Referencias de cada Df
# Referencias de morasDF
morasDF['Referencia'] = morasDF['Referencia'].apply(lambda x: refChangesDict.get(x, x))
# Referencias de Cartera
carteraDF['Referencia'] = carteraDF['Referencia'].apply(lambda x: refChangesDict.get(x, x))
# Referencias de mensualidadesDF
mensualidadesDF['Referencia'] = mensualidadesDF['Referencia'].apply(lambda x: refChangesDict.get(x, x))

## 2.2 **Filtrado de Referencias en morasDF**

In [ ]:
# Se van a dejar las Mensualidades y carteras de solo las Referencias que se encuentran en morasDF
refsMoras = morasDF['Referencia'].tolist()
carteraDF = carteraDF[carteraDF['Referencia'].isin(refsMoras)]
mensualidadesDF = mensualidadesDF[mensualidadesDF['Referencia'].isin(refsMoras)]

# Ahora para carteraDF se van a dejar los Datos donde la Fecha de Origen coincida con la Fecha de Origen de morasDF para cada Referencia
# Primero Creamos un Diccionario con la Fecha de Origen de morasDF por Referencia
fechaOrigenMorasDict = morasDF.set_index('Referencia')['Fecha_Origen'].to_dict()
# Luego se filtra carteraDF comparando la Fecha de Origen con la Fecha de Origen de morasDF para cada Referencia
carteraDF = carteraDF[(
    carteraDF.apply(
        lambda row: row['Fecha_Origen'].date() == fechaOrigenMorasDict.get(row['Referencia'].date(), None), axis=1
        ))]

print('\n✅Filtrado de Mensualidades Realizado con Éxito')
print('🚹Filas Totales: {}'.format(len(mensualidadesDF)))
print('\n✅Filtrado de Cartera Realizado con Éxito')
print('🚹Filas Totales: {}'.format(len(carteraDF)))

# 3. **Escritura de Datos a Local**
Los Datos que se van a guardar son:
- **morasDF**: Datos de los Estructurados
- **carteraDF**: Datos del Flujo de Estructurado
- **mensualidadesDF**: Datos de las Mensualidades por Cobrar

La razón del cambio se debe a que son flujos aparte, su relación de Unión debe ser exclusiva al momento de la creación de un nuevo flujo.

In [ ]:
# Carpeta donde se guardarán los archivos
outputDir = Path("data")

# Crear la carpeta si no existe
outputDir.mkdir(parents=True, exist_ok=True)

# Guardar archivos
# Path / Filename es igual a: data/moras.csv y data/moras.parquet
morasDF.to_parquet(outputDir / "moras.parquet", index=False)

# Ahora Realizamos lo mismo con carteraDF
# Path / Filename es igual a: data/cartera.csv y data/cartera.parquet
carteraDF.to_parquet(outputDir / "cartera.parquet", index=False)

# Ahora Realizamos lo mismo con mensualidadesDF
# Path / Filename es igual a: data/mensualidades.csv y data/mensualidades.parquet
mensualidadesDF.to_parquet(outputDir / "mensualidades.parquet", index=False)

# Imprimimos Ubicación de Datos
print("✅ DataFrames guardados en:")
print(outputDir / "moras.parquet")
print(outputDir / "cartera.parquet")
print(outputDir / "mensualidades.parquet")

# Imprimimos tamaño de datos
print("\n📊 Tamaño de los DataFrames:")
print(f"Moras en Cartera: {len(morasDF)} filas")
print(f"Cartera: {len(carteraDF)} filas")
print(f"Mensualidades: {len(mensualidadesDF)} filas")

# Finalizamos Proceso
print('\n✅Proceso de Pre-Cálculo Finalizado con Éxito')